# Step 7 – RAG with Foundry Agent Service (Portal-Visible)

This notebook creates a **service-managed Prompt Agent** in Azure AI Foundry Agent Service using the `azure-ai-projects` SDK.  
Unlike Steps 5–6 (which ran entirely locally), this agent is **registered server-side** and visible in the Foundry portal.

**Architecture:**
1. `AIProjectClient` → `agents.create_version()` creates a Prompt Agent with a `search_books` function tool
2. `project.get_openai_client()` → `responses.create()` sends prompts to the agent
3. When the agent calls `search_books`, we **execute the function locally** and return the result
4. The agent uses the search results to compose its final answer

**Key difference from Step 5/6:** The agent definition (name, instructions, tools) lives in the Foundry service.  
Function execution still happens locally — the portal can *see* the agent but can’t *run* custom functions.

In [1]:
%pip install azure-ai-projects>=2.0.0 azure-identity azure-search-documents openai python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup clients

In [24]:
import os
import json

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool, Tool
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery, QueryType
from openai import OpenAI
from openai.types.responses.response_input_param import FunctionCallOutput
from dotenv import load_dotenv

load_dotenv(override=False)

# ── Foundry project config ──
PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL = os.getenv("FOUNDRY_MODEL")
AGENT_NAME = "HarryPotterRAG"

# ── Project client (creates agents, gets OpenAI client) ──
credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
openai_client = project.get_openai_client()

# ── Embedding client (sync OpenAI SDK for hybrid search) ──
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
EMBEDDING_DIMS = 256
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
embedding_client = OpenAI(
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=token_provider,
)

# ── Azure AI Search client ──
search_client = SearchClient(
    endpoint=os.getenv("AZURE_SEARCH_ENDPOINT"),
    index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "rag-index"),
    credential=credential,
)

print(f"Project endpoint: {PROJECT_ENDPOINT}")
print(f"Model: {MODEL}")
print(f"Embedding model: {EMBEDDING_DEPLOYMENT} (dims={EMBEDDING_DIMS})")
print(f"Search index: {os.getenv('AZURE_SEARCH_INDEX_NAME')}")
print("Ready!")

Project endpoint: https://admin-megt79wf-eastus2.services.ai.azure.com/api/projects/admin-megt79wf-eastus2-project
Model: gpt-4.1
Embedding model: text-embedding-3-large-460208 (dims=256)
Search index: rag-index
Ready!


## Define the search function

This is the local function that the agent will call. It performs hybrid search + semantic reranking.

In [25]:
def search_books(query: str, top_k: int = 5) -> str:
    """Hybrid search + semantic reranking over the Harry Potter book index."""
    # Embed the query
    response = embedding_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=[query],
        dimensions=EMBEDDING_DIMS,
    )
    query_vector = response.data[0].embedding

    # Hybrid search with semantic reranking
    results = search_client.search(
        search_text=query,
        vector_queries=[
            VectorizedQuery(vector=query_vector, k=top_k, fields="embedding")
        ],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="default-semantic",
        top=top_k,
    )

    # Format results
    chunks = []
    for r in results:
        score_info = f"(reranker: {r.get('@search.reranker_score', 0):.2f})"
        chunks.append(f"[Page {r['page_number']}] {score_info}\n{r['content']}")

    if not chunks:
        return "No relevant passages found."

    return "\n\n---\n\n".join(chunks)

print("search_books function defined.")

search_books function defined.


## Create the Prompt Agent in Foundry

This registers the agent server-side. You'll be able to see it in the Foundry portal under **Agents**.

In [26]:
# Define the function tool schema for the agent
search_tool = FunctionTool(
    name="search_books",
    description="Search the Harry Potter books for information. Returns relevant passages with page numbers.",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find relevant passages in the Harry Potter books."
            },
            "top_k": {
                "type": "integer",
                "description": "Number of results to return.",
                "default": 5
            }
        },
        "required": ["query", "top_k"],
        "additionalProperties": False
    },
    strict=True,
)

tools: list[Tool] = [search_tool]

# Create (or update) the agent in Foundry
agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a helpful assistant that answers questions about the Harry Potter books. "
            "Use the search_books tool to find relevant passages before answering. "
            "Always cite page numbers in your answer like (Page X). "
            "If the search results don't contain enough information, say so."
        ),
        tools=tools,
    ),
)

print(f"Agent created! name={agent.name}, version={agent.version}, id={agent.id}")
print("\n→ Go to the Foundry portal to see this agent under 'Agents'.")

Agent created! name=HarryPotterRAG, version=2, id=HarryPotterRAG:2

→ Go to the Foundry portal to see this agent under 'Agents'.


## Helper: run a question with function-call handling

The Responses API returns `function_call` items when the agent wants to call our tool.  
We execute the function locally and submit the output back.

In [27]:
# Map of tool names to local functions
TOOL_FUNCTIONS = {
    "search_books": search_books,
}

def ask_agent(question: str, previous_response_id: str | None = None) -> tuple[str, str]:
    """Send a question to the agent, handle function calls, return (answer, last_response_id)."""
    # Build the request — chain via previous_response_id for multi-turn
    kwargs = dict(
        input=question,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
    if previous_response_id is not None:
        kwargs["previous_response_id"] = previous_response_id

    response = openai_client.responses.create(**kwargs)

    # Loop to handle function calls (may need multiple rounds)
    max_rounds = 5
    for _ in range(max_rounds):
        function_calls = [item for item in response.output if item.type == "function_call"]
        if not function_calls:
            break

        # Execute each function call and collect outputs
        tool_outputs = []
        for call in function_calls:
            func = TOOL_FUNCTIONS.get(call.name)
            if func is None:
                tool_outputs.append(FunctionCallOutput(
                    type="function_call_output",
                    call_id=call.call_id,
                    output=json.dumps({"error": f"Unknown function: {call.name}"}),
                ))
                continue

            args = json.loads(call.arguments)
            result = func(**args)
            tool_outputs.append(FunctionCallOutput(
                type="function_call_output",
                call_id=call.call_id,
                output=result,
            ))

        # Submit function results back
        response = openai_client.responses.create(
            input=tool_outputs,
            previous_response_id=response.id,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )

    return response.output_text, response.id

print("ask_agent helper defined.")

ask_agent helper defined.


## Ask questions

In [13]:
answer, _ = ask_agent(
    "What are the different names that the Dark Lord had in the book? "
    "Tell me in which parts of the book these names come up as well."
)
print(answer)

Here are some of the different names used for the Dark Lord in the Harry Potter books, along with where they appear:

1. **The Dark Lord**  
   - This is one of the most common titles used by Death Eaters, his followers, and even some others who fear him. It appears in many parts of the books, such as when Bellatrix and Narcissa talk with Snape (Page 2436, 2437, 2439, 2441, 2442), when Snape and Dumbledore discuss his plans (Page 3558), and when the Taboo on his name is mentioned (Page 3358).
   - For example, Snape says, “My information has been conveyed directly to the Dark Lord” (Page 2439).  
   - The name “Dark Lord” also comes up when Death Eaters discuss the taboo placed on saying his name (Page 3358).

2. **Lord Voldemort**  
   - This is his self-given name, used when he is referred to formally or with his chosen title. For example, Harry says, “… and Voldemort was there… Lord Voldemort…” (Page 1509), and Dumbledore refers to “… the plan Lord Voldemort is revolving around me” 

In [14]:
answer, _ = ask_agent(
    "In Chamber of Secrets, how is Voldemort's name and how does he name himself?"
)
print(answer)

In Harry Potter and the Chamber of Secrets, it is revealed that Voldemort's real name is Tom Marvolo Riddle. During a confrontation in the Chamber, Tom Riddle shows Harry how he rearranged the letters of his name to create his new identity, writing:

TOM MARVOLO RIDDLE  
and, with a wave of Harry's wand, the letters are magically rearranged to spell:  
I AM LORD VOLDEMORT

Riddle tells Harry, "It was a name I was already using at Hogwarts, to my most intimate friends only, of course. You think I was going to use my filthy Muggle father’s name forever? ... I fashioned myself a new name, a name I knew wizards everywhere would one day fear to speak" (Page 542).

This scene explains both the origin of Voldemort’s name and his deliberate rejection of his Muggle-heritage in crafting his new wizarding identity.


In [ ]:
answer, _ = ask_agent("Who is Harry's godfather and how is he related to his parents?")
print(answer)

## Multi-turn conversation

Reuse the same `conversation_id` for follow-up questions.

In [28]:
# Turn 1
answer, last_id = ask_agent("What is the Philosopher's Stone?")
print(f"Turn 1:\n{answer}\n")

# Turn 2 — follow-up chained via previous_response_id
answer, last_id = ask_agent("Who was trying to steal it and why?", previous_response_id=last_id)
print(f"Turn 2:\n{answer}\n")

Turn 1:
The Philosopher's Stone is a legendary magical object from ancient alchemy that can transform any metal into pure gold and also produces the Elixir of Life, which makes the drinker immortal. In the Harry Potter books, the only known Philosopher's Stone belongs to Nicolas Flamel, a famous alchemist who lives a long life with the help of the Stone (Page 198). 

If you need more detail or information about the Stone’s role in the story, let me know!

Turn 2:
Professor Quirrell was the one trying to steal the Philosopher's Stone. However, he was not acting alone—he was working under the influence and orders of Lord Voldemort, who had possessed his body. Voldemort wanted the Stone because it could grant him immortal life through the Elixir of Life, allowing him to regain his physical form and power (Page 1494, Page 263).



## Cleanup

Delete the agent version from Foundry when done (it will disappear from the portal too).

In [29]:
project.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print(f"Deleted agent '{agent.name}' version {agent.version} from Foundry.")

Deleted agent 'HarryPotterRAG' version 2 from Foundry.
